#### Overfitting and Leakage

This notebook presents examples of deliberate overfitting and data leakage to demonstrate how these issues lead to misleading metrics, and then shows how to fix them.

In [12]:
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

In [13]:
X, y = make_regression(n_samples=80, n_features=5, noise=15.0, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)

shallow_tree = DecisionTreeRegressor(max_depth=3, random_state=42)
deep_tree = DecisionTreeRegressor(max_depth=None, random_state=42)

shallow_tree.fit(X_train, y_train)
deep_tree.fit(X_train, y_train)

for name, model in [("shallow", shallow_tree), ("deep", deep_tree)]:
    train_rmse = mean_squared_error(y_train, model.predict(X_train))
    test_rmse = mean_squared_error(y_test, model.predict(X_test))
    print(f"{name} tree")
    print(f"Train RMSE: {train_rmse:.2f}")
    print(f"Test RMSE: {test_rmse:.2f}")
    print()

shallow tree
Train RMSE: 1413.31
Test RMSE: 10361.68

deep tree
Train RMSE: 0.00
Test RMSE: 9788.34



The deep tree has a train error of 0.0 but almost no improvement on the test set—a clear sign of overfitting.

#### Leakage

In [14]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X, y = make_classification(n_samples=200, n_features=5, n_informative=3, 
n_redundant=0, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42,
stratify=y)

# create a leaking version of a feature thet uses target
X_train_leaky = np.hstack([X_train, y_train.reshape(-1, 1)])
X_test_leaky = np.hstack([X_test, y_test.reshape(-1, 1)])

clf_leaky = LogisticRegression(max_iter=1000)
clf_leaky.fit(X_train_leaky, y_train)

y_pred_leaky = clf_leaky.predict(X_test_leaky)
acc_leaky = accuracy_score(y_test, y_pred_leaky)
print(f"Leaky classifier accuracy: {acc_leaky:.3f}")


Leaky classifier accuracy: 0.990


In [15]:
clf_clean = LogisticRegression(max_iter=1000)
clf_clean.fit(X_train, y_train)

y_pred_clean = clf_clean.predict(X_test)
acc_clean = accuracy_score(y_test, y_pred_clean)
print(f"Clean classifier accuracy: {acc_clean:.3f}")


Clean classifier accuracy: 0.890


#### Takeaways

**Overfitting** is when a model fits the training data so closely that it fails to generalize, as with the deep tree whose train RMSE was ~0 while test RMSE stayed almost as high as the shallow tree.

**Leakage** is when information from the target (or from the future) sneaks into the features, as when we stacked `y` into `X` and the “leaky” classifier jumped to 0.990 accuracy versus 0.890 for the clean one.

**Checks I will apply:**
- *Against overfitting:* always compare train vs test error (or use CV); a huge train–test gap means simplify or regularize before celebrating the train score.
- *Against leakage:* fit and transform only on the training split, and never build features from `y` (or any column that encodes the label) before evaluating on held-out data.